In [1]:
# pip install pyautogen
# ollama pull qwen3:1.7b
# %pip install "autogen-agentchat" - 0.7.5
# !pip install -U "ag2-classic[ollama]" agentops -0.4.21
# %pip install -U "autogen[ollama]" - 0.14.1

In [2]:
# Import libraries
import autogen
import os
import pprint

from pathlib import Path

from autogen import ConversableAgent
from autogen.coding import CodeBlock, LocalCommandLineCodeExecutor


In [3]:
# import agentops
# agentops.init("yourkey")

In [4]:
# Create working directory and executor

work_dir = Path("coding")
work_dir.mkdir(exist_ok = True)
executor = LocalCommandLineCodeExecutor(work_dir=work_dir)

print("Working directory created:", work_dir)
print("Code executor initialized successfully!")


# Create Agent 1 – Code Executor Agent

code_executor_agent = ConversableAgent(name = "code_executor_agent", llm_config = False,
                                       code_execution_config={"executor": executor},
                                       human_input_mode = "NEVER")

print("Code Executor Agent created successfully!")

Working directory created: coding
Code executor initialized successfully!
Code Executor Agent created successfully!


In [5]:
# Configure Ollama LLM

config_list = [{"model": "qwen3:1.7b", "api_type": "ollama", "stream": False, "temperature": 0}]

In [6]:
# Define Code Writer System Message

code_writer_system_message = """
You are an expert Python coding assistant.

Your responsibility is to solve programming tasks using Python.

Instructions:

1. Understand the given task carefully.
2. Explain your plan briefly.
3. Generate optimized and executable Python code.
4. Use Python code blocks when providing code.
5. Avoid incomplete code.
6. Execute the code through the code executor agent.
7. Check the execution result.
8. Provide the final answer clearly.

The user cannot modify the generated code.

Always print the required result using the print() function.
"""

# Create Agent 2 – Code Writer Agent

code_writer_agent = ConversableAgent(name = "code_writer", system_message = code_writer_system_message,
                                     llm_config = {"config_list": config_list}, code_execution_config = False,
                                     max_consecutive_auto_reply = 2, human_input_mode = "NEVER")

print("Code Writer Agent created successfully!")

Code Writer Agent created successfully!


In [7]:
# Start communication between agents

chat_result = code_executor_agent.initiate_chat(code_writer_agent, 
                                                message = """
                                                Write optimized Python code to calculate the number
                                                of unique permutations of the word 'ALGEBRA'.
                                                The word ALGEBRA contains repeated letters.
                                                Use efficient logic to calculate the unique permutations.
                                                Verify the result using the word ALGEBRA.
                                                The code should only return the final permutation count.
                                                """)
pprint.pprint(chat_result)

code_executor_agent (to code_writer):


                                                Write optimized Python code to calculate the number
                                                of unique permutations of the word 'ALGEBRA'.
                                                The word ALGEBRA contains repeated letters.
                                                Use efficient logic to calculate the unique permutations.
                                                Verify the result using the word ALGEBRA.
                                                The code should only return the final permutation count.
                                                

--------------------------------------------------------------------------------
code_writer (to code_executor_agent):

To calculate the number of unique permutations of the word **'ALGEBRA'**, we use the formula for permutations of a multiset:

$$
\text{Unique Permutations} = \frac{n!}{k_1! \cdot k_2! \cdots k_m!}
$$

Wh

In [8]:
chat_result.cost

{'usage_including_cached_inference': {'total_cost': 0.0,
  'qwen3:1.7b': {'cost': 0.0,
   'prompt_tokens': 676,
   'completion_tokens': 1292,
   'total_tokens': 1968}},
 'usage_excluding_cached_inference': {'total_cost': 0.0,
  'qwen3:1.7b': {'cost': 0.0,
   'prompt_tokens': 676,
   'completion_tokens': 1292,
   'total_tokens': 1968}}}

In [9]:
chat_result.chat_id

87932851153616016980835845401692352094

In [10]:
chat_result.summary

'exitcode: 0 (execution succeeded)\nCode output: 2520\n'

In [ ]:
# qwen3:1.7b - 2.0B
# nomic-embed-text:latest - 137M
# llama3.2:latest - 3.2B